# MongoDB ML Pipeline

## 1. Setup

In [ ]:
!pip -q install -r requirements.txt


## 2. Load Environment

In [ ]:
import os
from dotenv import load_dotenv
load_dotenv()
print("MONGODB_URI set:", bool(os.getenv("MONGODB_URI")))
print("MONGODB_DB:", os.getenv("MONGODB_DB", "FARSI"))
print("MONGODB_COLLECTION:", os.getenv("MONGODB_COLLECTION", "crime_events"))


## 3. Ingest CSV to MongoDB

In [ ]:
from pipeline.ingest_crime_csv import ingest_csv
inserted = ingest_csv("data/crime/2025-11-avon-and-somerset-street.csv")
print("Inserted:", inserted)


## 4. Train Model from MongoDB

In [ ]:
from pipeline.train_crime_model import train_and_save
meta = train_and_save(output_dir="models")
meta["best_model"]


## 5. Load Model and Predict

In [ ]:
import joblib
import pandas as pd
from pipeline.mongo_client import get_collection
from pipeline.feature_engineering import clean_and_engineer

collection = get_collection()
data = list(collection.find({}, {"crime_type": 1, "month": 1, "location": 1, "latitude": 1, "longitude": 1, "reported_by": 1, "falls_within": 1, "lsoa_code": 1, "lsoa_name": 1, "last_outcome_category": 1, "context": 1}).limit(20))
df = pd.DataFrame(data)
df = clean_and_engineer(df)

model = joblib.load("models/crime_type_model.joblib")
preds = model.predict(df)
preds
